In [ ]:
from pathlib import Path
import json, subprocess, sys, zipfile, shutil
INPUT=Path('/kaggle/input'); WORK=Path('/kaggle/working')
m=list(INPUT.rglob('active_tb_safe_eval_v1/EVAL_BOUNDARY.json'))
if len(m)==1: SRC=m[0].parent
else:
 a=list(INPUT.rglob('active_tb_safe_eval_v1.zip')); assert len(a)==1,a
 d=WORK/'att'
 with zipfile.ZipFile(a[0]) as zf: zf.extractall(d)
 SRC=d/'active_tb_safe_eval_v1'
sz=SRC/'suites'
if not sz.is_dir():
 sz=WORK/'suites'
 with zipfile.ZipFile(SRC/'suites.zip') as zf: zf.extractall(sz)
print(json.loads((SRC/'EVAL_BOUNDARY.json').read_text()))
print('suites:',len(list(sz.glob('*.v'))))
subprocess.run(['apt-get','update','-qq'],check=True)
subprocess.run(['apt-get','install','-y','-qq','iverilog'],check=True)


In [ ]:
OUT=WORK/'safe_eval'; OUT.mkdir(exist_ok=True)
r=subprocess.run([sys.executable,str(SRC/'sealed_dual_eval.py'),
 '--suites',str(sz),'--manifest',str(SRC/'safe_manifest.json'),
 '--checkers',str(SRC/'checkers.jsonl'),'--golden',str(SRC/'golden.jsonl'),
 '--mutants',str(SRC/'hidden_bank_a.jsonl'),
 '--out',str(OUT/'dual_records.csv')],text=True,capture_output=True)
print(r.stdout[-6000:])
if r.returncode: print('STDERR:',r.stderr[-3000:])
shutil.make_archive(str(WORK/'safe_eval'),'zip',OUT)
print('packaged')
